### Grammatiken

[Videos](https://youtube.com/playlist?list=PLWeMgMhRDsIFXFROQfwE51uALeTmAnnK_&si=mOaNCc4DZvOmMmf4) - [Aufgabenvideo](https://youtu.be/fx1UbrnnnSo)

Dieses Notebook setzt das Notebook **Reguläre Sprachen** voraus: Alphabet, Wort, Sprache, endliche Automaten und
reguläre Ausdrücke. Dort haben wir Sprachen *erkannt*: Ein Automat liest ein Wort und entscheidet, ob es dazugehört.
Eine Grammatik geht den umgekehrten Weg: Sie *erzeugt* die Wörter einer Sprache mit Ersetzungsregeln.

#### Syntax und Semantik

Bei einer Sprache unterscheidet man

- die **Syntax**: Welche Zeichenfolgen sind korrekt gebildet? Das ist die Form.
- die **Semantik**: Was bedeutet eine korrekt gebildete Zeichenfolge? Das ist der Inhalt.

Der Satz *Farblose grüne Ideen schlafen wütend* (nach einem Beispiel von Noam Chomsky) ist grammatisch richtig, hat
aber keinen vernünftigen Sinn. *Schlafen Ideen grüne wütend farblose* ist schon syntaktisch falsch.

Bei Programmiersprachen ist es genauso:

In [1]:
programme = ["print('Hallo'", "x = 1 / 0", "x = 2 +* 3", "x = len(5)"]
for p in programme:
    try:
        compile(p, '<test>', 'exec')
        print(f'{p:14} syntaktisch korrekt')
    except SyntaxError:
        print(f'{p:14} Syntaxfehler')

print('Hallo'  Syntaxfehler
x = 1 / 0      syntaktisch korrekt
x = 2 +* 3     Syntaxfehler
x = len(5)     syntaktisch korrekt


`compile` prüft nur die Syntax. `x = 1 / 0` und `x = len(5)` sind syntaktisch korrekt, ihre Bedeutung führt aber erst
beim Ausführen zu einem Fehler (Division durch null, `len` einer Zahl). Grammatiken beschreiben nur die Syntax.

#### Formale Grammatik

Eine **formale Grammatik** besteht aus

1. $V$, einer endlichen Menge von **Nichtterminalen** (auch Variablen),
2. $\Sigma$, einem endlichen Alphabet mit $\Sigma \cap V = \emptyset$. Die Zeichen aus $\Sigma$ heißen **Terminale**,
3. $P$, einer endlichen Menge von **Produktionsregeln** der Form $u \rightarrow v$,
4. $S \in V$, dem **Startsymbol**.

Nichtterminale schreiben wir als Großbuchstaben, Terminale als Kleinbuchstaben oder Ziffern.

Man beginnt mit $S$ und ersetzt so lange mit den Regeln, bis nur noch Terminale dastehen. Die Zwischenschritte
heißen **Satzformen**. Die Folge der Ersetzungen ist eine **Ableitung**, wir schreiben sie mit $\Rightarrow$. Die
**Sprache der Grammatik** $L(G)$ ist die Menge aller Wörter aus Terminalen, die sich aus $S$ ableiten lassen.

#### Beispiel

$V = \{S\}$, $\Sigma = \{0, 1\}$, Startsymbol $S$, Produktionsregeln: $S \rightarrow 01$, $S \rightarrow 0S1$

Regeln mit gleicher linker Seite fasst man mit $\mid$ zusammen: $S \rightarrow 01 \mid 0S1$.

Eine Ableitung: $S \Rightarrow 0S1 \Rightarrow 00S11 \Rightarrow 000111$

Jede Anwendung von $S \rightarrow 0S1$ setzt eine $0$ vorne und eine $1$ hinten dazu, zum Schluss beendet
$S \rightarrow 01$ die Ableitung. Die Grammatik erzeugt also

$$L = \{0^n1^n \mid n \ge 1\}$$

Das leere Wort ist nicht dabei, weil jede Ableitung mit $S \rightarrow 01$ endet. Mit den Regeln
$S \rightarrow 0S1 \mid \varepsilon$ erhält man $\{0^n1^n \mid n \ge 0\}$, also auch $\varepsilon$.

Aus dem Notebook **Reguläre Sprachen** wissen wir: Diese Sprache ist nicht regulär. Grammatiken können also mehr
beschreiben als endliche Automaten.

#### Übung

Leite das Wort $aababa$ aus der folgenden Grammatik ab:

$V = \{S, A, B\}$, $\Sigma = \{a, b\}$, Startsymbol $S$

$P$: $S \rightarrow AB$, $A \rightarrow aS \mid a$, $B \rightarrow bA$

<details><summary>Lösung</summary>

$S \Rightarrow AB \Rightarrow aSB \Rightarrow aABB \Rightarrow aaBB \Rightarrow aabAB \Rightarrow aabaB
\Rightarrow aababA \Rightarrow aababa$

Hier wurde immer das am weitesten links stehende Nichtterminal ersetzt. Das nennt man eine **Linksableitung**.
</details>

#### Ableitungsbaum

Ein **Ableitungsbaum** zeigt, welches Nichtterminal durch welche rechte Seite ersetzt wurde. Die Wurzel ist $S$,
die Kinder eines Knotens sind die Zeichen der rechten Seite. Die Blätter ergeben von links nach rechts gelesen das
Wort.

<img src="baum_aababa.png" width="280">

Der Baum hängt nicht davon ab, in welcher Reihenfolge man die Nichtterminale ersetzt. Zu jedem Baum gehört genau eine
Linksableitung.

#### Grammatiken in Python

Wir speichern die Regeln als Dictionary: Zu jedem Nichtterminal gehört die Liste seiner rechten Seiten. Das leere
Wort $\varepsilon$ ist der leere String.

In [2]:
regeln = {'S': ['0S1', '01']}

def erstes_nt(form):
    '''Position des ersten Nichtterminals (Großbuchstabe), None wenn es keins gibt'''
    for i, z in enumerate(form):
        if z.isupper():
            return i
    return None

def ersetzungen(regeln, form):
    '''alle Satzformen, die durch Ersetzen des ersten Nichtterminals entstehen'''
    i = erstes_nt(form)
    return [form[:i] + rechts + form[i + 1:] for rechts in regeln[form[i]]]

ersetzungen(regeln, '00S11')

['000S111', '000111']

Alle Wörter bis zu einer Länge $n$ findet man mit einer Suche über die Satzformen. Terminale verschwinden nie wieder,
deshalb kann man Satzformen mit mehr als $n$ Terminalen weglassen. Damit die Suche auch bei Regeln wie
$A \rightarrow \varepsilon$ endet, begrenzen wir zusätzlich die Länge der Satzformen.

In [3]:
def woerter(regeln, n):
    '''alle Wörter mit höchstens n Zeichen, die aus S ableitbar sind'''
    ergebnis = set()
    offen = ['S']
    gesehen = {'S'}
    while offen:
        form = offen.pop()
        if erstes_nt(form) is None:
            ergebnis.add(form)
            continue
        for neu in ersetzungen(regeln, form):
            terminale = sum(1 for z in neu if not z.isupper())
            if terminale <= n and len(neu) <= 2 * n + 4 and neu not in gesehen:
                gesehen.add(neu)
                offen.append(neu)
    return sorted(ergebnis, key=lambda w: (len(w), w))

woerter(regeln, 8)

['01', '0011', '000111', '00001111']

Das **Wortproblem** fragt: Gehört ein gegebenes Wort zur Sprache? Bei unseren Grammatiken kann man alle Wörter bis
zur Länge des Wortes erzeugen und nachsehen. Für eine Ableitung merken wir uns bei der Suche, aus welcher Satzform
jede neue Satzform entstanden ist (Breitensuche).

In [4]:
def ableitung(regeln, wort):
    '''eine Linksableitung des Wortes als Liste von Satzformen, None wenn es keine gibt'''
    vorher = {'S': None}
    offen = ['S']
    while offen:
        naechste = []
        for form in offen:
            if form == wort:
                weg = []
                while form is not None:
                    weg.append(form)
                    form = vorher[form]
                return weg[::-1]
            if erstes_nt(form) is None:
                continue
            for neu in ersetzungen(regeln, form):
                i = erstes_nt(neu)
                anfang = neu if i is None else neu[:i]
                if neu not in vorher and wort.startswith(anfang) and len(neu) <= 2 * len(wort) + 4:
                    vorher[neu] = form
                    naechste.append(neu)
        offen = naechste
    return None

regeln = {'S': ['AB'], 'A': ['aS', 'a'], 'B': ['bA']}
print(' => '.join(ableitung(regeln, 'aababa')))
print(ableitung(regeln, 'abab'))

S => AB => aSB => aABB => aaBB => aabAB => aabaB => aababA => aababa
None


Die Terminale vor dem ersten Nichtterminal ändern sich bei einer Linksableitung nicht mehr. Passen sie nicht zum
Anfang des Wortes, kann man die Satzform weglassen.

Welche Wörter erzeugt diese Grammatik?

In [5]:
woerter(regeln, 9)

['aba',
 'aababa',
 'abaaba',
 'aaabababa',
 'aabaababa',
 'aababaaba',
 'abaaababa',
 'abaabaaba']

Jedes $S$ wird zu $AB$, also $A\,b\,A$, und jedes $A$ zu einem $a$, eventuell mit einem neuen $S$ dahinter. Pro $b$
entstehen damit genau zwei $a$: In jedem Wort kommt $a$ doppelt so oft vor wie $b$. Umgekehrt ist nicht jedes solche
Wort dabei, z.B. fehlt $aab$, denn jedes Wort endet mit $a$.

#### Grammatiken entwerfen

Einige Bausteine kommen immer wieder vor:

| Ziel | Regeln | erzeugt |
|---|---|---|
| beliebig viele $a$ | $A \rightarrow aA \mid \varepsilon$ | $a^n$, $n \ge 0$ |
| mindestens ein $a$ | $A \rightarrow aA \mid a$ | $a^n$, $n \ge 1$ |
| erst das eine, dann das andere | $S \rightarrow AB$ | Wörter aus $A$ gefolgt von Wörtern aus $B$ |
| eines von beiden | $S \rightarrow A \mid B$ | Wörter aus $A$ oder aus $B$ |
| gleich viele, von außen nach innen | $S \rightarrow aSb \mid \varepsilon$ | $a^nb^n$ |

Beispiel: Gesucht ist eine Grammatik für $\{a^jb^kc^l \mid j, k, l \ge 0,\ j > l\}$.

Zu jedem $c$ gehört ein $a$, das erzeugen wir von außen nach innen mit $A \rightarrow aAc$. Dazu kommen beliebig viele
weitere $a$ ($A \rightarrow aA$), mindestens eins davon ($S \rightarrow aA$), und in der Mitte beliebig viele $b$.

$S \rightarrow aA$ <br>
$A \rightarrow aA \mid aAc \mid B$ <br>
$B \rightarrow bB \mid \varepsilon$

Ob eine Grammatik stimmt, kann man in Python für kurze Wörter nachprüfen: Man vergleicht die erzeugten Wörter mit
allen Wörtern, die die Bedingung erfüllen.

In [6]:
import itertools

def alle_woerter(alphabet, n):
    for laenge in range(n + 1):
        for t in itertools.product(alphabet, repeat=laenge):
            yield ''.join(t)

def bedingung(w):
    j, k, l = w.count('a'), w.count('b'), w.count('c')
    return w == 'a' * j + 'b' * k + 'c' * l and j > l

regeln = {'S': ['aA'], 'A': ['aA', 'aAc', 'B'], 'B': ['bB', '']}
erzeugt = set(woerter(regeln, 8))
soll = {w for w in alle_woerter('abc', 8) if bedingung(w)}
print(len(erzeugt), len(soll), erzeugt == soll)

70 70 True


### Reguläre Grammatiken

Eine Grammatik heißt **regulär** (oder rechtslinear), wenn jede Regel die Form

$$A \rightarrow aB \qquad A \rightarrow a \qquad A \rightarrow \varepsilon$$

hat: Auf der rechten Seite steht höchstens ein Terminal, und ein Nichtterminal darf nur ganz rechts stehen.

Reguläre Grammatiken erzeugen genau die regulären Sprachen. Zu einem DEA bekommt man die Grammatik so:

- Jeder Zustand wird ein Nichtterminal, der Startzustand das Startsymbol $S$.
- Aus dem Übergang $q \xrightarrow{a} r$ wird die Regel $Q \rightarrow aR$.
- Für jeden Endzustand $q$ kommt $Q \rightarrow \varepsilon$ dazu.

Beispiel: der DEA für die Wörter über $\{0, 1\}$ ohne $11$ aus dem Notebook **Reguläre Sprachen**, mit $S$ für $q_0$,
$A$ für $q_1$. Der Fehlerzustand $q_2$ erzeugt kein Wort und kann wegfallen.

$S \rightarrow 0S \mid 1A \mid \varepsilon$ <br>
$A \rightarrow 0S \mid \varepsilon$

Eine Ableitung entspricht genau einem Lauf durch den Automaten: Das Nichtterminal am Ende der Satzform ist der
aktuelle Zustand.

In [7]:
regeln = {'S': ['0S', '1A', ''], 'A': ['0S', '']}
print(' => '.join(ableitung(regeln, '10100')))
erzeugt = set(woerter(regeln, 10))
soll = {w for w in alle_woerter('01', 10) if '11' not in w}
print(len(erzeugt), erzeugt == soll)

S => 1A => 10S => 101A => 1010S => 10100S => 10100
375 True


Umgekehrt wird aus einer regulären Grammatik ein NEA: Nichtterminale werden Zustände, $A \rightarrow aB$ wird ein
Übergang von $A$ nach $B$ mit $a$, $A \rightarrow \varepsilon$ macht $A$ zum Endzustand, und für $A \rightarrow a$
braucht man einen zusätzlichen Endzustand.

### Kontextfreie Grammatiken

Eine Grammatik heißt **kontextfrei**, wenn auf jeder linken Seite genau ein Nichtterminal steht. Rechts ist alles
erlaubt. Kontextfrei heißt: $A$ darf ersetzt werden, egal was links und rechts von $A$ steht.

Alle Grammatiken, die wir in Python als Dictionary geschrieben haben, sind kontextfrei. Eine Sprache heißt
**kontextfrei**, wenn es eine kontextfreie Grammatik für sie gibt.

Beispiele, die nicht regulär sind:

- $\{0^n1^n \mid n \ge 0\}$: $S \rightarrow 0S1 \mid \varepsilon$
- Palindrome über $\{a, b\}$: $S \rightarrow aSa \mid bSb \mid a \mid b \mid \varepsilon$
- korrekt geklammerte Ausdrücke wie `(()(()))`: $S \rightarrow (S)S \mid \varepsilon$

In [8]:
regeln = {'S': ['(S)S', '']}
print(woerter(regeln, 6))
print(' => '.join(ableitung(regeln, '(()())')))

['', '()', '(())', '()()', '((()))', '(()())', '(())()', '()(())', '()()()']
S => (S)S => ((S)S)S => (()S)S => (()(S)S)S => (()()S)S => (()())S => (()())


#### Arithmetische Ausdrücke und Mehrdeutigkeit

Die Grammatik

$E \rightarrow E+E \mid E*E \mid Z$ <br>
$Z \rightarrow 2 \mid 3 \mid 4$

erzeugt Ausdrücke wie $2+3*4$. Für dieses Wort gibt es aber zwei verschiedene Ableitungsbäume:

<table><tr>
<td><img src="baum_mehrdeutig1.png" width="230"></td>
<td><img src="baum_mehrdeutig2.png" width="230"></td>
</tr></table>

Liest man den Baum als Rechenweg (erst die Teilbäume ausrechnen, dann den Knoten), ergibt der linke $2 + 12 = 14$
und der rechte $5 \cdot 4 = 20$. Eine Grammatik mit zwei Ableitungsbäumen für dasselbe Wort heißt **mehrdeutig**. Für
eine Programmiersprache ist das unbrauchbar.

Die folgende Grammatik erzeugt dieselben Ausdrücke (mit Klammern), ist aber eindeutig. Sie baut die Regel *Punkt vor
Strich* ein: Ein Ausdruck $E$ ist eine Summe von Termen $T$, ein Term ein Produkt von Faktoren $F$.

$E \rightarrow E+T \mid T$ <br>
$T \rightarrow T*F \mid F$ <br>
$F \rightarrow (E) \mid Z$ <br>
$Z \rightarrow 2 \mid 3 \mid 4$

<img src="baum_eindeutig.png" width="260">

Das $*$ hängt tiefer im Baum als das $+$, also wird zuerst multipliziert.

### Die Chomsky-Hierarchie

Noam Chomsky hat 1956 Grammatiken nach der Form ihrer Regeln eingeteilt:

| Typ | Name | Regeln | Automat |
|---|---|---|---|
| 0 | allgemein | keine Einschränkung | Turingmaschine |
| 1 | kontextsensitiv | $u \rightarrow v$ mit $\lvert u \rvert \le \lvert v \rvert$ | linear beschränkter Automat |
| 2 | kontextfrei | $A \rightarrow v$ | Kellerautomat |
| 3 | regulär | $A \rightarrow aB \mid a \mid \varepsilon$ | endlicher Automat |

Bei Typ 1 dürfen Satzformen also nie kürzer werden. Damit trotzdem $\varepsilon$ zur Sprache gehören kann, ist als
Ausnahme $S \rightarrow \varepsilon$ erlaubt, wenn $S$ auf keiner rechten Seite vorkommt.

Jede Grammatik vom Typ 3 ist auch vom Typ 2, jede vom Typ 2 auch vom Typ 1 (bis auf Regeln mit $\varepsilon$, die man
entfernen kann) und jede vom Typ 1 auch vom Typ 0. Eine Sprache hat den Typ $i$, wenn es eine Grammatik vom Typ $i$ für
sie gibt. Jede Stufe enthält echt mehr Sprachen als die darunter:

- $\{0^n1^n\}$ ist kontextfrei, aber nicht regulär.
- $\{a^nb^nc^n \mid n \ge 1\}$ ist kontextsensitiv, aber nicht kontextfrei (siehe Notebook **Kellerautomaten**).
- Die Menge der Programme, die anhalten, ist vom Typ 0, aber nicht vom Typ 1 (siehe Notebook **Halteproblem**).

#### Eine kontextsensitive Grammatik für $a^nb^nc^n$

$S \rightarrow aSBC \mid aBC$ <br>
$CB \rightarrow BC$ <br>
$aB \rightarrow ab \qquad bB \rightarrow bb \qquad bC \rightarrow bc \qquad cC \rightarrow cc$

$S$ erzeugt $a^n$ und dahinter $n$-mal $B$ und $C$ durcheinander. $CB \rightarrow BC$ sortiert alle $B$ vor die $C$,
danach werden sie von links nach rechts zu Kleinbuchstaben. Links stehen hier auch zwei Zeichen, deshalb speichern
wir die Regeln als Liste von Paaren und ersetzen an jeder Stelle, nicht nur beim ersten Nichtterminal.

In [9]:
regeln = [('S', 'aSBC'), ('S', 'aBC'), ('CB', 'BC'), ('aB', 'ab'), ('bB', 'bb'), ('bC', 'bc'), ('cC', 'cc')]

def woerter_typ1(regeln, n):
    ergebnis = set()
    offen = ['S']
    gesehen = {'S'}
    while offen:
        form = offen.pop()
        if not any(z.isupper() for z in form):
            ergebnis.add(form)
            continue
        for links, rechts in regeln:
            i = form.find(links)
            while i >= 0:
                neu = form[:i] + rechts + form[i + len(links):]
                if len(neu) <= n and neu not in gesehen:
                    gesehen.add(neu)
                    offen.append(neu)
                i = form.find(links, i + 1)
    return sorted(ergebnis, key=len)

woerter_typ1(regeln, 12)

['abc', 'aabbcc', 'aaabbbccc', 'aaaabbbbcccc']

Weil Satzformen bei Typ 1 nie kürzer werden, darf die Suche alle Satzformen weglassen, die länger als $n$ sind. So
findet sie sicher alle Wörter bis zur Länge $n$, und das Wortproblem ist für Typ 1 entscheidbar. Bei Typ 0 geht das
nicht, dort können Satzformen erst wachsen und dann wieder schrumpfen.

### EBNF

Die Syntax von Programmiersprachen wird meist mit kontextfreien Grammatiken beschrieben. Man schreibt sie in der
**erweiterten Backus-Naur-Form (EBNF)**:

- Statt $\rightarrow$ schreibt man `::=`.
- Terminale stehen in Hochkommata, z.B. `'a'`. Nichtterminale dürfen ganze Wörter sein, z.B. `Zahl`.
- Wiederholungen und Optionen kann man direkt hinschreiben:

| EBNF | Bedeutung |
|---|---|
| `A?` | $A$ oder nichts (optional) |
| `A*` | beliebig oft $A$, auch keinmal |
| `A+` | mindestens einmal $A$ |
| `A B` | $A$ gefolgt von $B$ |
| `A \| B` | $A$ oder $B$ |
| `( ... )` | Klammern zum Gruppieren |
| `[0-9]` | ein Zeichen aus dem Bereich |

Es gibt mehrere Varianten der EBNF. Wir verwenden die Schreibweise des Railroad Diagram Generators (siehe unten).

Beispiele:

```
Ziffer    ::= [0-9]
Zahl      ::= '-'? Ziffer+
Kommazahl ::= Zahl ( '.' Ziffer+ )?
Name      ::= ( Buchstabe | '_' ) ( Buchstabe | Ziffer | '_' )*
```

Die Regel für `Name` beschreibt, welche Variablennamen in Python erlaubt sind (nur mit den Buchstaben a bis z, A bis Z).

Die Grammatik zur Übung oben sieht in EBNF so aus. Die Regel für $B$ wird zu `'b'*`:

```
S ::= 'a' A
A ::= 'a' A 'c'? | 'b'*
```

Kommt in einer EBNF keine Rekursion vor, dann kann man alle Regeln ineinander einsetzen und erhält einen regulären
Ausdruck. Pythons `re` versteht dieselben Zeichen `?`, `*`, `+`, `|` und `[0-9]`:

In [10]:
import re

ziffer = '[0-9]'
zahl = f'-?{ziffer}+'
kommazahl = f'{zahl}(\\.{ziffer}+)?'
buchstabe = '[a-zA-Z]'
name = f'({buchstabe}|_)({buchstabe}|{ziffer}|_)*'

for w in ['42', '-7', '3.14', '-0.5', '3.', '.5', '1-2']:
    print(f'{w:5}', bool(re.fullmatch(kommazahl, w)))
print()
for w in ['summe', '_x2', '2x', 'max_wert', 'a-b']:
    print(f'{w:9}', bool(re.fullmatch(name, w)))

42    True
-7    True
3.14  True
-0.5  True
3.    False
.5    False
1-2   False

summe     True
_x2       True
2x        False
max_wert  True
a-b       False


Bei rekursiven Regeln wie `Klammer ::= ( '(' Klammer ')' )*` geht das nicht: Korrekt geklammerte Ausdrücke sind
nicht regulär.

### Syntaxdiagramme

Ein **Syntaxdiagramm** (auch Eisenbahndiagramm) stellt eine Regel als Gleisplan dar. Man fährt von links nach rechts
und darf an jeder Weiche wählen. Jede Fahrt von links nach rechts ergibt ein erlaubtes Wort.

- Abgerundete Kästen sind **Terminale**: Dieses Zeichen wird geschrieben.
- Rechteckige Kästen sind **Nichtterminale**: Hier fährt man durch das Diagramm mit diesem Namen und kommt danach
  zurück.

Die Bausteine der EBNF:

<table>
<tr><td><code>A ::= 'a'?</code></td><td><img src="sd_optional.png" width="170"></td>
<td><code>A ::= 'a'*</code></td><td><img src="sd_stern.png" width="170"></td></tr>
<tr><td><code>A ::= 'a'+</code></td><td><img src="sd_plus.png" width="170"></td>
<td><code>A ::= 'a' | B</code></td><td><img src="sd_oder.png" width="170"></td></tr>
</table>

Beim Stern läuft man das Gleis mit $a$ beliebig oft rückwärts im Kreis oder fährt gleich oben vorbei. Beim Plus muss
man mindestens einmal durch das $a$.

#### Zahlen und Namen

<img src="sd_ziffer.png" width="150"> <img src="sd_zahl.png" width="250">

<img src="sd_kommazahl.png" width="400">

<img src="sd_name.png" width="420">

#### Ausdrücke

Rekursion sieht man daran, dass ein Diagramm über andere Diagramme wieder sich selbst benutzt: `Faktor` enthält
`Ausdruck`.

```
Ausdruck ::= Term ( '+' Term )*
Term     ::= Faktor ( '*' Faktor )*
Faktor   ::= Zahl | '(' Ausdruck ')'
```

<img src="sd_ausdruck.png" width="330">

<img src="sd_term.png" width="330">

<img src="sd_faktor.png" width="330">

Das ist dieselbe Sprache wie bei der eindeutigen Grammatik oben, nur ohne Linksrekursion: Statt $E \rightarrow E+T$
heißt es *ein Term, dann beliebig oft $+$ und ein Term*.

#### Übung

Gib zur Sprache $\{a^jb^kc^l \mid j, k, l \ge 0,\ j > l\}$ eine EBNF und die Syntaxdiagramme an.

<details><summary>Lösung</summary>

```
S ::= 'a' A
A ::= 'a' A 'c'? | 'b'*
```

<img src="sd_jkl_s.png" width="160">

<img src="sd_jkl_a.png" width="260">
</details>

#### Railroad Diagram Generator

Der [Railroad Diagram Generator](https://www.bottlecaps.de/rr/ui) zeichnet zu einer eingegebenen EBNF die
Syntaxdiagramme. Man gibt die Regeln unter *Edit Grammar* ein und sieht die Diagramme unter *View Diagram*.

### Vom Syntaxdiagramm zum Programm

Ein Programm, das die Syntax eines Textes prüft, heißt **Parser**. Jeder Compiler und jeder Interpreter beginnt damit.
Aus Syntaxdiagrammen bekommt man einen Parser fast direkt: Jedes Diagramm wird eine Funktion, ein Nichtterminal ein
Funktionsaufruf, eine Schleife im Diagramm eine `while`-Schleife und eine Weiche ein `if`. Weil `faktor` wieder
`ausdruck` aufruft, heißt das Verfahren **rekursiver Abstieg**.

Jede Funktion bekommt den Text und die aktuelle Position und gibt den Wert des erkannten Teils und die neue Position
zurück. So rechnet der Parser den Ausdruck gleich aus.

In [11]:
def ausdruck(s, i):
    wert, i = term(s, i)
    while i < len(s) and s[i] == '+':
        w, i = term(s, i + 1)
        wert = wert + w
    return wert, i

def term(s, i):
    wert, i = faktor(s, i)
    while i < len(s) and s[i] == '*':
        w, i = faktor(s, i + 1)
        wert = wert * w
    return wert, i

def faktor(s, i):
    if i < len(s) and s[i] == '(':
        wert, i = ausdruck(s, i + 1)
        if i < len(s) and s[i] == ')':
            return wert, i + 1
        raise SyntaxError(f"')' fehlt an Stelle {i}")
    return zahl(s, i)

def zahl(s, i):
    anfang = i
    while i < len(s) and s[i].isdigit():
        i = i + 1
    if i == anfang:
        raise SyntaxError(f'Zahl erwartet an Stelle {i}')
    return int(s[anfang:i]), i

def auswerten(s):
    wert, i = ausdruck(s, 0)
    if i < len(s):
        raise SyntaxError(f'unerwartetes Zeichen an Stelle {i}')
    return wert

In [12]:
for s in ['2+3*4', '(2+3)*4', '10*(1+2*3)+5', '2+*3', '(2+3', '2+3)']:
    try:
        print(f'{s:13}', auswerten(s))
    except SyntaxError as e:
        print(f'{s:13}', 'Syntaxfehler:', e)

2+3*4         14
(2+3)*4       20
10*(1+2*3)+5  75
2+*3          Syntaxfehler: Zahl erwartet an Stelle 2
(2+3          Syntaxfehler: ')' fehlt an Stelle 4
2+3)          Syntaxfehler: unerwartetes Zeichen an Stelle 3


Punkt vor Strich ergibt sich von selbst, weil `term` die Multiplikationen erledigt, bevor `ausdruck` addiert.
Pythons eigener Parser ist komplizierter, beruht aber auf demselben Prinzip. Die Python-Dokumentation beschreibt die
Syntax der Sprache ebenfalls mit Regeln in einer EBNF-ähnlichen Schreibweise.

Nicht alles lässt sich mit einer kontextfreien Grammatik ausdrücken. Dass eine Variable vor ihrer Verwendung einen
Wert bekommen hat, prüft keine Grammatik. Solche Fehler fallen in Python erst beim Ausführen auf.

### Zusammenfassung

- Die **Syntax** legt fest, welche Zeichenfolgen korrekt gebildet sind, die **Semantik**, was sie bedeuten.
- Eine **Grammatik** besteht aus Nichtterminalen, Terminalen, Produktionsregeln und einem Startsymbol. Ihre Sprache
  sind alle Wörter aus Terminalen, die sich aus dem Startsymbol ableiten lassen.
- Ein **Ableitungsbaum** zeigt die Struktur eines Wortes. Gibt es für ein Wort zwei verschiedene Bäume, ist die
  Grammatik **mehrdeutig**.
- **Chomsky-Hierarchie:** regulär (Typ 3) $\subset$ kontextfrei (Typ 2) $\subset$ kontextsensitiv (Typ 1)
  $\subset$ allgemein (Typ 0). Reguläre Grammatiken entsprechen endlichen Automaten, kontextfreie den
  Kellerautomaten.
- **EBNF** und **Syntaxdiagramme** beschreiben kontextfreie Grammatiken kompakt, mit `?`, `*`, `+` und `|` bzw. mit
  Weichen und Schleifen. Aus Syntaxdiagrammen lässt sich ein Parser mit rekursivem Abstieg programmieren.